# 02 — Exponential Smoothing

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapter 3.

Exponential smoothing solves the flat-weighting problem of moving average.
It gives **exponentially decreasing weight** to older observations — the most
recent period matters most, but older periods still contribute.

$$f_{t+1} = \alpha \cdot d_t + (1 - \alpha) \cdot f_t$$

Where `α` (alpha) controls the smoothing:
- **α close to 1** → reacts fast, more weight on recent demand
- **α close to 0** → smooth, slow to react, more weight on history


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

## 1. The Model

In [ ]:
def exp_smoothing(d, extra_periods=12, alpha=0.4):
    """
    Simple exponential smoothing (SES).

    Parameters
    ----------
    d : array-like — historical demand
    extra_periods : int — future periods to forecast
    alpha : float in (0,1) — smoothing parameter

    Returns
    -------
    pd.DataFrame with columns: Demand, Forecast, Error
    """
    cols = len(d)
    d = np.append(d, [np.nan] * extra_periods)
    f = np.full(cols + extra_periods, np.nan)

    # Initialise: first forecast = first demand observation
    f[1] = d[0]

    for t in range(2, cols):
        f[t] = alpha * d[t - 1] + (1 - alpha) * f[t - 1]

    # Future forecast = flat (SES has no trend)
    f[cols:] = f[cols - 1]

    return pd.DataFrame({'Demand': d, 'Forecast': f, 'Error': d - f})

## 2. Simulated Demand — with a demand level shift

In [ ]:
np.random.seed(42)
# Demand shifts upward at period 24 — simulating a product launch effect
demand = np.concatenate([
    np.random.normal(1000, 80, 24),
    np.random.normal(1400, 80, 24)
])
demand = np.clip(demand, 0, None)

## 3. Compare alpha values

In [ ]:
df_low  = exp_smoothing(demand, extra_periods=6, alpha=0.1)
df_mid  = exp_smoothing(demand, extra_periods=6, alpha=0.4)
df_high = exp_smoothing(demand, extra_periods=6, alpha=0.8)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_mid['Demand'], label='Actual Demand', color='#2C2C2A', linewidth=1.5)
ax.plot(df_low['Forecast'],  label='α=0.1 (slow)', color='#D85A30', linestyle='--')
ax.plot(df_mid['Forecast'],  label='α=0.4 (balanced)', color='#185FA5', linestyle='--')
ax.plot(df_high['Forecast'], label='α=0.8 (fast)', color='#1D9E75', linestyle='--')
ax.axvline(x=len(demand), color='gray', linestyle=':', alpha=0.5, label='Forecast horizon')
ax.set_title('Exponential Smoothing: Effect of Alpha on Demand Level Shift', fontsize=13)
ax.set_xlabel('Period (months)')
ax.set_ylabel('Demand (units)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Optimise Alpha Automatically

Use `scipy.optimize` to find the alpha that minimises MAE on historical data.

In [ ]:
def mae_for_alpha(alpha, demand):
    df = exp_smoothing(demand, extra_periods=0, alpha=alpha[0])
    return np.mean(np.abs(df['Error'].dropna()))

result = minimize(
    mae_for_alpha,
    x0=[0.5],
    args=(demand,),
    bounds=[(0.01, 0.99)]
)

optimal_alpha = result.x[0]
print(f'Optimal alpha: {optimal_alpha:.3f}')
print(f'Minimised MAE: {result.fun:.1f} units')

## 5. Key Takeaways

- SES is better than moving average because it weights recent observations more heavily
- Alpha is a hyperparameter — optimise it on historical data
- SES still produces a **flat future forecast** — no trend, no seasonality

**Next:** Double exponential smoothing (`03_double_exponential_smoothing.ipynb`) adds a trend component.
